In [51]:
import pandas as pd
import csv
import numpy as np

pd.set_option("future.no_silent_downcasting", True)

In [58]:
# import dialogue data
dialogue_raw = pd.read_csv(
    "dialogue/movie_lines.tsv", 
    sep="\t", 
    header=None, 
    names=["lineID", "characterID", "movieID", "character", "text"],
    quoting=csv.QUOTE_NONE,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)

# import movie data
movies_raw = pd.read_csv(
    "dialogue/movie_titles_metadata.tsv", 
    sep="\t", 
    header=None, 
    names=["movieID", "title", "year", "imdb_rating", "imdb_votes", "genres"],
    quoting=csv.QUOTE_NONE,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)

# import character data
character_raw = pd.read_csv(
    "dialogue/movie_characters_metadata.tsv", 
    sep="\t", 
    header=None, 
    names=["characterID", "character", "movieID", "title", "gender", "position"],
    quoting=csv.QUOTE_NONE,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)   

# import Oscar nominees data
oscars_raw = pd.read_csv("oscars/the_oscar_award.csv", header=0)

In [ ]:
# clean Oscars data
oscars = oscars_raw[["year_film", "film", "canon_category", "winner"]].copy()
oscars.dropna(subset=["film"], inplace=True)
oscars.rename(
    columns={
        "year_film": "year",
        "film": "title",
        "canon_category": "award"
    }, inplace=True)
oscars["title"] = oscars["title"].str.replace(r'[^A-Za-z0-9 ]', '', regex=True).str.lower().str.strip()
oscars["award"] = oscars["award"].astype("category")

# aggregate oscar awards data
oscars = oscars.groupby(["title", "year"]).agg({
    "award": lambda x: len(x.unique())
}).reset_index()

# clean dialogue data
dialogue = dialogue_raw.copy()
dialogue.dropna(subset=["text"], inplace=True)
dialogue["text"] = dialogue["text"].str.lower().str.strip()
dialogue["text"] = dialogue["text"].str.replace(r'[^\w\s]', '', regex=True)
dialogue["movieID"] = dialogue["movieID"].str.replace(r'[^0-9]', '', regex=True)
dialogue["movieID"] = dialogue["movieID"].astype("Int64")

# clean movie data
movies = movies_raw[["movieID", "title", "year", "genres"]].copy()
movies["year"] = movies["year"].str.replace(r'[^0-9 ]', '', regex=True).str.strip()
movies["year"] = movies["year"].astype("Int64")
movies["title"] = movies["title"].str.replace(r'[^A-Za-z0-9 ]', '', regex=True).str.lower().str.strip()
movies["movieID"] = movies["movieID"].str.replace(r'[^0-9]', '', regex=True)
movies["movieID"] = movies["movieID"].astype("Int64")

# # clean character data
# characters = character_raw["characterID"].copy()

In [79]:
# join the dataframes
df = dialogue.merge(movies, on="movieID", how="left")
df = df.merge(oscars, on=["title", "year"], how="left")
# df = df.merge(characters, on="characterID", how="left")

print(df.isnull().sum())

lineID              0
characterID         0
movieID             0
character          43
text                0
tokens              0
title               0
year                0
genres              0
award          177876
dtype: int64


movies as a whole
vs movies with oscar nominations
vs movies without oscar nominations

In [80]:
import nltk
from nltk.tokenize import word_tokenize
# nltk.download("punkt_tab")

# tokenize dialogue text 
df["tokens"] = df["text"].str.lower().apply(word_tokenize)

# drop stop words
stop_words = set(nltk.corpus.stopwords.words("english"))
df["tokens"] = df["tokens"].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [82]:
# split dataframes for movies with and without Oscar nominations
noms = df[df["award"].notna()].copy()
no_noms = df[df["award"].isna()].copy()

In [83]:
# movies with at least one Oscar nomination
fdist_noms = nltk.FreqDist(noms["tokens"].explode())

fdist_noms = {word: freq for word, freq in fdist_noms.items() if word not in stop_words}

# filter out shorter words
fdist_noms = {word: freq for word, freq in fdist_noms.items() if len(str(word)) >= 4}

# get most common words in nominated movies
top_noms = dict(sorted(fdist_noms.items(), key=lambda item: item[1], reverse=True)[:100])
# top_noms

In [84]:
# movies with no Oscar nominations
fdist_none = nltk.FreqDist(no_noms["tokens"].explode())

# drop stop words
fdist_none = {word: freq for word, freq in fdist_none.items() if word not in stop_words}

# filter out shorter words
fdist_none = {word: freq for word, freq in fdist_none.items() if len(str(word)) >= 4}

# get most common words in movies with no nominations
top_none = dict(sorted(fdist_none.items(), key=lambda item: item[1], reverse=True)[:100])
# top_none

In [85]:
## find common words in both sets
# common_words = set(top_noms.keys()) & set(top_none.keys())
# common_words

# find unique words in each set
unique_noms = set(top_noms.keys()) - set(top_none.keys())
unique_none = set(top_none.keys()) - set(top_noms.keys())
unique_noms, unique_none

({'aint',
  'every',
  'fine',
  'hear',
  'listen',
  'made',
  'thank',
  'three',
  'understand',
  'wouldnt'},
 {'dead',
  'fuck',
  'fucking',
  'guys',
  'kill',
  'might',
  'shit',
  'stop',
  'talking',
  'wait'})

In [ ]:
cfd = nltk.ConditionalFreqDist(
    (genre, word)
    for genre in dialogue["genres"].dropna().explode() 
    for word in dialogue["tokens"].explode())

NLP of dialogue responses Oscar noms vs none